# AI-Powered Document Analysis System (RAG)

**Stack:** Python · LangChain · FAISS · Local LLM (Hugging Face / Ollama) · Streamlit

This notebook is the **development workspace** for an end-to-end
retrieval-augmented generation (RAG) pipeline over PDF documents. It:

1. Loads and chunks source PDFs
2. Embeds chunks and builds a FAISS vector index
3. Loads a local LLM (no OpenAI/Anthropic API calls — fully offline once models are cached)
4. Assembles a retrieval + generation chain with LangChain
5. Runs a question against the pipeline with source citations
6. Benchmarks retrieval relevance against a labeled query set

Once the logic here is validated, it is refactored into standalone modules
under `src/` (`document_loader.py`, `vector_store.py`, `llm_handler.py`,
`rag_pipeline.py`, `evaluate.py`) plus a `config.py` and a Streamlit app
(`app.py`) — see the accompanying project README for the final structure.


## 0. Setup

Install dependencies (run once).

In [ ]:
# !pip install -r requirements.txt
# If running standalone, uncomment the line above.


In [ ]:
import json
from pathlib import Path

import numpy as np

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

DATA_DIR = Path("data/pdfs")
INDEX_DIR = Path("vector_index")
BENCHMARK_PATH = Path("benchmark/benchmark_queries.json")

DATA_DIR.mkdir(parents=True, exist_ok=True)
INDEX_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 1000
CHUNK_OVERLAP = 150
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
HF_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"   # small instruct model, CPU/GPU friendly
TOP_K = 4


## 1. (Optional) Generate sample PDFs

If you don't have PDFs handy yet, generate two small sample documents
so the pipeline can be exercised end-to-end. Skip this cell and drop
your own PDFs into `data/pdfs/` instead.


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/create_sample_data.py"], check=True)
list(DATA_DIR.glob("*.pdf"))


## 2. Load & chunk PDFs

Each PDF page becomes a LangChain `Document`. Pages are then split into
overlapping ~1000-character chunks — small enough for precise retrieval,
with enough overlap that an answer split across a chunk boundary isn't lost.


In [ ]:
def load_pdfs(pdf_dir=DATA_DIR):
    pdf_paths = sorted(Path(pdf_dir).glob("*.pdf"))
    if not pdf_paths:
        raise FileNotFoundError(f"No PDFs found in {pdf_dir}")
    documents = []
    for path in pdf_paths:
        pages = PyPDFLoader(str(path)).load()
        for page in pages:
            page.metadata["source"] = path.name
        documents.extend(pages)
    print(f"Loaded {len(documents)} pages from {len(pdf_paths)} PDF(s).")
    return documents

raw_docs = load_pdfs()


In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(raw_docs)
for i, c in enumerate(chunks):
    c.metadata["chunk_id"] = i

print(f"Split into {len(chunks)} chunks.")
print("\nSample chunk:\n", chunks[0].page_content[:300])
print("\nMetadata:", chunks[0].metadata)


## 3. Embed chunks & build the FAISS index

`all-MiniLM-L6-v2` is a small, fast sentence-embedding model (384-dim)
that runs comfortably on CPU. Embeddings are normalized so cosine
similarity search behaves consistently.


In [ ]:
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

vector_store = FAISS.from_documents(chunks, embeddings)
print(f"FAISS index built with {vector_store.index.ntotal} vectors.")

vector_store.save_local(str(INDEX_DIR))
print(f"Saved index to {INDEX_DIR}")


In [ ]:
# Sanity check: does semantic search return sensible chunks?
test_results = vector_store.similarity_search("home office stipend amount", k=2)
for r in test_results:
    print(f"[{r.metadata['source']} | page {r.metadata['page']}]")
    print(r.page_content[:200], "\n")


## 4. Load the local LLM

Using a small instruction-tuned model (`Qwen2.5-1.5B-Instruct`) via
Hugging Face `transformers`, wrapped as a LangChain `HuggingFacePipeline`
so it can be composed inside an LCEL chain. Swap `HF_MODEL_NAME` for a
larger model if you have the GPU memory, or switch to Ollama (see
`src/llm_handler.py` for the alternate backend).


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(HF_MODEL_NAME, device_map="auto")

text_gen_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.2,
    do_sample=True,
    return_full_text=False,
)

llm = HuggingFacePipeline(pipeline=text_gen_pipeline)


## 5. Assemble the RAG chain

Retriever → prompt (with retrieved context injected) → local LLM → parsed
string output. This mirrors exactly what `src/rag_pipeline.py` does, just
inline for experimentation.


In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": TOP_K})

SYSTEM_PROMPT = (
    "You are a precise research assistant. Answer the question using ONLY "
    "the provided context extracted from the user's PDF documents. If the "
    "answer is not contained in the context, say you don't know instead of "
    "guessing. Be concise."
)

PROMPT_TEMPLATE = """{system_prompt}

Context:
{context}

Question: {question}

Answer:"""

prompt = PromptTemplate(
    template=PROMPT_TEMPLATE,
    input_variables=["system_prompt", "context", "question"],
)

def format_docs(docs):
    return "\n\n".join(
        f"[Source: {d.metadata.get('source','?')} | page {d.metadata.get('page','?')}]\n{d.page_content}"
        for d in docs
    )

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
        "system_prompt": lambda _: SYSTEM_PROMPT,
    }
    | prompt
    | llm
    | StrOutputParser()
)


## 6. Ask a question

In [ ]:
question = "How much is the home office stipend and what is the receipt deadline?"

retrieved = retriever.invoke(question)
answer = rag_chain.invoke(question)

print("QUESTION:", question)
print("\nANSWER:\n", answer.strip())
print("\nSOURCES USED:")
for d in retrieved:
    print(f"- {d.metadata['source']} (page {d.metadata['page']}): {d.page_content[:120]}...")


## 7. Benchmark evaluation

Loads `benchmark/benchmark_queries.json` (hand-labeled questions with
expected keywords + a short reference answer) and scores each query on:

- **Keyword coverage** — fraction of expected keywords present in the
  retrieved chunks
- **Semantic similarity** — max cosine similarity between the reference
  answer and any retrieved chunk (same embedding model used for indexing)

A query is marked *relevant* if either signal clears its threshold. The
percentage of benchmark queries marked relevant is the headline metric
reported in the resume bullet ("**>90% relevance on benchmark queries**").


In [ ]:
KEYWORD_COVERAGE_THRESHOLD = 0.6
SIMILARITY_THRESHOLD = 0.45

def keyword_coverage(text, keywords):
    if not keywords:
        return 0.0
    text_lower = text.lower()
    hits = sum(1 for kw in keywords if kw.lower() in text_lower)
    return hits / len(keywords)

def cosine_sim(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / denom) if denom else 0.0

with open(BENCHMARK_PATH) as f:
    benchmark = json.load(f)

results = []
for item in benchmark:
    q = item["question"]
    keywords = item.get("expected_keywords", [])
    reference = item.get("reference_answer", "")

    docs = retriever.invoke(q)
    combined = "\n".join(d.page_content for d in docs)
    kw_score = keyword_coverage(combined, keywords)

    max_sim = 0.0
    if reference:
        ref_vec = np.array(embeddings.embed_query(reference))
        for d in docs:
            sim = cosine_sim(ref_vec, np.array(embeddings.embed_query(d.page_content)))
            max_sim = max(max_sim, sim)

    is_relevant = kw_score >= KEYWORD_COVERAGE_THRESHOLD or max_sim >= SIMILARITY_THRESHOLD
    results.append((q, kw_score, max_sim, is_relevant))

print(f"{'Question':<55} {'KW':>6} {'Sim':>6} {'Relevant?':>10}")
print("-" * 80)
for q, kw, sim, rel in results:
    q_disp = (q[:52] + "...") if len(q) > 52 else q
    print(f"{q_disp:<55} {kw:>6.2f} {sim:>6.2f} {'YES' if rel else 'no':>10}")

relevance_pct = 100 * sum(r[3] for r in results) / len(results)
print("-" * 80)
print(f"Overall relevance: {relevance_pct:.1f}% across {len(results)} benchmark queries")


## 8. Next: modularize into `src/`

The logic above is now split into standalone files so it can be reused
by both a CLI and the Streamlit app, without re-running notebook cells:

| Notebook section | Extracted to |
|---|---|
| Config constants (§0) | `src/config.py` |
| Load & chunk PDFs (§2) | `src/document_loader.py` |
| Embeddings + FAISS (§3) | `src/vector_store.py` |
| Local LLM loading (§4) | `src/llm_handler.py` |
| RAG chain + query (§5, §6) | `src/rag_pipeline.py` |
| Benchmark evaluation (§7) | `src/evaluate.py` |
| — | `app.py` (Streamlit UI) |

See `README.md` for how to run the modular version and the Streamlit app.
